# Rol 4. Especialista en modelos generativos (Nadissa)

**Task 1.4.** En este notebook se evalúa si conviene generar secuencias sospechosas sintéticas para compensar los pocos positivos. Se comparan tres generadores y se redacta la sección 4 del memo.

Se agrega la raíz del repositorio al `sys.path` para importar los módulos de `src/`. Los datos del enunciado están en `src/case_data.py`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import case_data as cd

## 1. Cobertura de tipologías

Se calcula la `cobertura` como la fracción de las 7 tipologías de lavado que cubre cada generador. El costo de difusión (3 días) se expresa en horas.

In [2]:
gens = cd.generators()
gens["cobertura"] = gens["tipologias_cubiertas"] / gens["tipologias_totales"]
gens.round(2)

,generador,tipologias_cubiertas,tipologias_totales,costo_horas,observacion,cobertura
0,GAN,2,7,4,Repite una sola forma de fraccionamiento,0.29
1,VAE,5,7,2,"Secuencias borrosas, promedia montos",0.71
2,Difusión,6,7,72,Mayor diversidad y nitidez,0.86


### a. Comportamiento de la GAN y del autoencoder variacional

**GAN (cobertura de 0.29).** El generador aprende a engañar al discriminador, no a reproducir toda la distribución. Cuando encuentra una forma de secuencia que el discriminador acepta, le basta repetirla. Este fenómeno se llama **colapso de modos** y explica que repita una sola forma de fraccionamiento y cubra solo 2 de 7 tipologías.

**Autoencoder variacional (cobertura de 0.71).** Se entrena con una pérdida de reconstrucción (error cuadrático) más un término de regularización KL. Cuando varias secuencias son posibles para un mismo punto del espacio latente, la reconstrucción que minimiza el error cuadrático es su **promedio**. Por eso las secuencias salen borrosas, con montos promediados y sin los patrones exactos. El término KL obliga a cubrir el espacio latente, lo que explica su mejor cobertura (5 de 7).

## 2. Efecto en el clasificador

In [3]:
gain = cd.AUC_TEMPORAL_DIFFUSION - cd.AUC_TEMPORAL_SPLIT
print(f"AUC temporal con sintéticos de difusión: {cd.AUC_TEMPORAL_SPLIT} -> {cd.AUC_TEMPORAL_DIFFUSION} (+{gain:.2f})")
print("AUC temporal con sintéticos de la GAN: sin cambio")

AUC temporal con sintéticos de difusión: 0.81 -> 0.83 (+0.02)
AUC temporal con sintéticos de la GAN: sin cambio


### b. Recomendación y validación

Se recomienda usar datos sintéticos **solo del modelo de difusión** y **solo después de mitigar el riesgo de privacidad**. Es el generador con mayor cobertura (0.86) y el único que mejora el clasificador, con un aumento de AUC temporal de 0.02 (de 0.81 a 0.83). La GAN no cambia el AUC y no se recomienda. La ganancia es pequeña frente a un costo de 72 horas, por lo que debe confirmarse antes de adoptarla.

La validación se hace así.

- **Entrenamiento.** Datos reales de enero a marzo más secuencias sintéticas generadas solo a partir de esos mismos meses.
- **Prueba.** Únicamente datos reales de abril y mayo. Las secuencias sintéticas nunca se usan para medir el desempeño.
- **Comparación.** El mismo clasificador entrenado solo con datos reales, evaluado en el mismo periodo, y el recall por tipología para confirmar que las tipologías agregadas se detectan.

### c. Mitigación del riesgo de privacidad

La auditoría encontró que el 3% de las secuencias de difusión es casi idéntico a secuencias de clientes reales, es decir, el modelo memorizó clientes. Se proponen dos medidas.

1. **Filtro de distancia.** Descartar toda secuencia sintética cuya distancia a la secuencia real más cercana sea menor que un umbral, y repetir la auditoría hasta que no queden casi duplicados.
2. **Entrenamiento con privacidad diferencial (DP-SGD).** Limita cuánto puede aprender el generador de un cliente individual y reduce la memorización desde el origen.

### d. Sección 4 del memo

> **4. Modelos generativos.** Se recomienda usar sintéticos solo del modelo de difusión, que cubre 6 de 7 tipologías y sube el AUC temporal de 0.81 a 0.83. La GAN se descarta por colapso de modos (2 de 7) y el autoencoder variacional por generar secuencias borrosas. Antes de usarlos se deben eliminar los casi duplicados de clientes reales (3% de los sintéticos) con un filtro de distancia. Los sintéticos se usan solo para entrenar y el desempeño se mide únicamente con datos reales de abril y mayo.